In [3]:
import pandas as pd
import numpy as np
import os
import uuid

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
OUTPUT_DIR = "nac_relational_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

POPULATION_TARGETS = {
    2025: 350000, 2030: 1050000, 2035: 1950000, 
    2040: 3100000, 2045: 4600000, 2050: 6500000
}

# GLOBAL MEMORY SETS TO PREVENT ID COLLISIONS
USED_NATIONAL_IDS = set()
USED_REAL_ESTATE_IDS = set()

GENDER_CHOICES = ['Male', 'Female']
GENDER_PROBS = [0.505, 0.495]

CAREERS = {
    'Civil Engineering': (120000, 700000), 'Accounting': (80000, 400000),
    'Medicine': (150000, 1200000), 'Software Engineering': (180000, 900000),
    'Architecture': (120000, 800000), 'Pharmacy': (90000, 450000),
    'Digital Marketing': (100000, 600000), 'Nursing': (80000, 300000),
    'Data Science': (200000, 1000000), 'Cybersecurity': (250000, 1100000),
    'Unemployed/Student': (0, 0), 'Retired': (40000, 240000)
}
MAJORS = [k for k in CAREERS.keys() if k not in ['Unemployed/Student', 'Retired']]
MAJOR_PROBS = [0.25, 0.20, 0.12, 0.10, 0.08, 0.08, 0.07, 0.05, 0.03, 0.02]

RESIDENCE_SITES = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
SITE_PROBS = [0.05, 0.10, 0.25, 0.10, 0.10, 0.05, 0.25, 0.10] 

HOSPITAL_SERVICES = ['None', 'Cardiovascular Services', 'Oncology Services', 'Neurology', 
                     'Physical Therapy', 'Specialized Medical Councils', 'Mental Health', 
                     'Ophthalmology', 'Specialized Dentistry']
CHRONIC_DISEASES = ['None', 'Type 2 Diabetes', 'Hypertension', 'Obesity', 'Asthma/Respiratory']
INSURANCE_TYPES = ['Universal Health Insurance (State)', 'Private Premium', 'Private Basic', 'Out-of-Pocket']

RESIDENCE_TYPES = ['Apartment', 'Villa', 'Twin House', 'Town House', 'Studio']
TRANSPORT_TYPES = ['Monorail', 'LRT', 'Fly Taxi', 'Electric Vehicle', 'Public Bus', 'Private Car (Combustion)', 'Active Commute (Bicycle/Walk)']

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def generate_real_estate_id():
    while True:
        re_id = ''.join([str(np.random.randint(0, 9)) for _ in range(14)])
        if re_id not in USED_REAL_ESTATE_IDS:
            USED_REAL_ESTATE_IDS.add(re_id)
            return re_id

def generate_national_id(yob, gender):
    century = "2" if yob < 2000 else "3"
    year_str = str(yob)[-2:]
    month = str(np.random.randint(1, 13)).zfill(2)
    day = str(np.random.randint(1, 29)).zfill(2)
    gov_code = str(np.random.choice([1, 21, 2, 3, 22, 88], p=[0.50, 0.30, 0.05, 0.05, 0.05, 0.05])).zfill(2)
    
    while True:
        base_seq = np.random.randint(100, 999)
        seq = base_seq if (gender == 'Male' and base_seq % 2 != 0) or (gender == 'Female' and base_seq % 2 == 0) else base_seq + 1
        check_digit = str(np.random.randint(1, 9))
        nat_id = f"{century}{year_str}{month}{day}{gov_code}{seq}{check_digit}"
        
        if nat_id not in USED_NATIONAL_IDS:
            USED_NATIONAL_IDS.add(nat_id)
            return nat_id

def assign_education(age):
    if age < 4: return "None", "None", "None", "None", "None"
    elif age <= 18:
        grades = ["Kindergarten", "Primary", "Preparatory", "Secondary"]
        idx = min(3, max(0, (age - 4) // 3))
        return grades[idx], "Basic", np.random.choice(["Public", "Private", "International"], p=[0.70, 0.25, 0.05]), "None", "None"
    else:
        uni_type = np.random.choice(["Public", "Private", "National", "International"], p=[0.65, 0.20, 0.10, 0.05])
        return "Graduate/Post-Graduate", "Higher Ed", "None", np.random.choice(MAJORS, p=MAJOR_PROBS), uni_type

def calculate_skewed_income(career):
    min_sal, max_sal = CAREERS.get(career, (0, 0))
    if max_sal == 0: return 0
    mode_sal = min_sal + ((max_sal - min_sal) * 0.2)
    return int(np.random.triangular(min_sal, mode_sal, max_sal))

def append_to_csv(df, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, mode='a', index=False, header=not os.path.exists(filepath), encoding='utf-8-sig')

# ==========================================
# POPULATION GENERATION
# ==========================================
def generate_population_batch(year, num_people):
    data = []
    num_families = max(1, int(num_people / 3.9)) 
    
    for _ in range(num_families):
        real_estate_id = generate_real_estate_id()
        residence_site = np.random.choice(RESIDENCE_SITES, p=SITE_PROBS)
        res_type = np.random.choice(RESIDENCE_TYPES, p=[0.60, 0.15, 0.10, 0.10, 0.05])
        family_size = max(1, int(np.random.normal(3.9, 1.2)))
        
        has_solar = np.random.choice([0, 1], p=[0.7, 0.3])
        has_ac = np.random.choice([0, 1], p=[0.05, 0.95])
        uses_greywater = np.random.choice([0, 1], p=[0.6, 0.4] if res_type in ['Villa', 'Twin House'] else [0.9, 0.1])
        
        opt_in_chance = 0.70 if res_type in ['Villa', 'Twin House'] else 0.25
        smart_grid_opt_in = 1 if np.random.random() < opt_in_chance else 0
        
        green_space_sqm = int(np.random.uniform(10, 150)) if res_type in ['Villa', 'Twin House', 'Town House'] else int(np.random.uniform(0, 15))
        
        elec_kw = np.random.normal(600, 150) if has_ac else np.random.normal(250, 80)
        if has_solar: elec_kw *= 0.35 
        gas_m3 = np.random.normal(35, 10)
        water_m3 = np.random.normal(25, 6) if uses_greywater else np.random.normal(35, 8)
        food_waste_kg = np.random.normal(15, 5) 
        
        for member_idx in range(family_size):
            if len(data) >= num_people: break 
                
            if member_idx == 0 and family_size > 1: age, gender, marital = int(max(21, np.random.normal(30.8, 3))), 'Male', 'Married'
            elif member_idx == 1 and family_size > 1: age, gender, marital = int(max(18, np.random.normal(25.0, 2.5))), 'Female', 'Married'
            elif family_size == 1: age, gender, marital = int(np.random.normal(28, 8)), np.random.choice(GENDER_CHOICES, p=GENDER_PROBS), 'Single'
            else: age, gender, marital = int(np.random.uniform(0, 15)), np.random.choice(GENDER_CHOICES, p=GENDER_PROBS), 'Single'
            
            yob = year - age
            national_id = generate_national_id(yob, gender)
            yod = yob + int(np.random.normal(74, 12))
            
            career = 'Unemployed/Student' if age < 18 else ('Retired' if age >= 65 else np.random.choice(MAJORS, p=MAJOR_PROBS))
            income = calculate_skewed_income(career)
            grade, ed_type, school_type, major, uni_type = assign_education(age)
            if age >= 18 and career not in ['Unemployed/Student', 'Retired']: major = career
            
            trans_type = np.random.choice(TRANSPORT_TYPES, p=[0.15, 0.15, 0.05, 0.20, 0.15, 0.20, 0.10])
            green_transit = 1 if trans_type in ['Monorail', 'LRT', 'Fly Taxi', 'Electric Vehicle', 'Public Bus', 'Active Commute (Bicycle/Walk)'] else 0
            
            if trans_type == 'Active Commute (Bicycle/Walk)':
                active_mins = int(np.random.normal(300, 50))
                bmi = round(np.random.normal(22, 2), 1)
            else:
                active_mins = int(max(0, np.random.normal(120, 60)))
                bmi = round(np.random.normal(26, 4), 1)
            
            if bmi > 30: chronic = np.random.choice(['Obesity', 'Type 2 Diabetes', 'Hypertension'], p=[0.4, 0.3, 0.3])
            else: chronic = np.random.choice(CHRONIC_DISEASES, p=[0.75, 0.05, 0.05, 0.05, 0.10])
            
            hosp_service = np.random.choice(HOSPITAL_SERVICES, p=[0.70, 0.05, 0.02, 0.02, 0.05, 0.01, 0.06, 0.05, 0.04])
            health_percent = int(np.random.uniform(50, 95)) if chronic != 'None' else int(np.random.uniform(85, 100))
            needs_bed = 1 if (hosp_service in ['Cardiovascular Services', 'Oncology Services', 'Neurology'] and np.random.random() < 0.2) else 0
            telemed_consults = int(max(0, np.random.normal(3, 2)))
            insurance = np.random.choice(INSURANCE_TYPES, p=[0.60, 0.10, 0.20, 0.10])
            
            base_recycling_chance = 0.8 if (has_solar and trans_type == 'Electric Vehicle') else 0.4
            segregates_waste = 1 if np.random.random() < base_recycling_chance else 0
            e_waste_kg = round(np.random.normal(10, 3) if career in ['Software Engineering', 'Data Science', 'Cybersecurity'] else np.random.normal(5, 2), 1)
            carbon_footprint = (elec_kw * 0.45) + (gas_m3 * 2.0) + (food_waste_kg * 2.5) - (segregates_waste * 15) - (green_transit * 20)
            
            data.append({
                'National_ID': national_id, 'Real_Estate_ID': real_estate_id,
                'Year_of_Birth': yob, 'Year_of_Death': yod, 'Gender': gender, 'Current_Age': age,
                'Marital_Status': marital, 'Career': career, 'Income_per_Year_EGP': income,
                'Edu_Grade': grade, 'Edu_Type': ed_type, 'School_Type': school_type, 'University_Major': major, 'University_Type': uni_type,
                'Residence_Site': residence_site, 'Residence_Type': res_type, 'Household_Green_Space_sqm': green_space_sqm,
                'Health_Percent': health_percent, 'Chronic_Disease': chronic, 'Hospital_Service_Needed': hosp_service, 
                'Requires_Hospital_Bed': needs_bed, 'BMI': bmi, 'Weekly_Active_Minutes': active_mins, 
                'Telemedicine_Consults_Per_Year': telemed_consults, 'Health_Insurance_Type': insurance,
                'Has_Solar_System': has_solar, 'Has_AC': has_ac, 'Uses_Greywater_Recycling': uses_greywater, 
                'Smart_Grid_Tariff_Opt_In': smart_grid_opt_in, 'Segregates_Waste_Recycling': segregates_waste, 'Annual_E_Waste_kg': e_waste_kg,
                'Avg_Monthly_Electricity_KW': round(elec_kw, 2), 'Avg_Monthly_Gas_m3': round(gas_m3, 2),
                'Avg_Monthly_Water_m3': round(water_m3, 2), 'Avg_Monthly_Food_Waste_kg': round(food_waste_kg, 2),
                'Carbon_Footprint_Index': round(carbon_footprint, 2),
                'Primary_Transport_Type': trans_type, 'Public_Transit_Green_Fuel_Usage': green_transit
            })
            
    return pd.DataFrame(data)

# ==========================================
# EXPORTING FACT & DIMENSION TABLES
# ==========================================
def extract_and_save_relational_data(df, is_new_batch, year):
    if is_new_batch:
        dim_resident = df[['National_ID', 'Gender', 'Year_of_Birth', 'Year_of_Death']].drop_duplicates(subset=['National_ID'])
        append_to_csv(dim_resident, 'Dim_Resident.csv')
        
        dim_real_estate = df[['Real_Estate_ID', 'Residence_Site', 'Residence_Type', 'Household_Green_Space_sqm']].drop_duplicates(subset=['Real_Estate_ID'])
        append_to_csv(dim_real_estate, 'Dim_Real_Estate.csv')

    df_facts = df.copy()
    df_facts['Report_Year'] = year
    df_facts['Record_ID'] = [str(uuid.uuid4()) for _ in range(len(df_facts))]
    
    fact_socio = df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Real_Estate_ID', 'Current_Age', 'Marital_Status', 
                           'Career', 'Income_per_Year_EGP', 'Primary_Transport_Type', 'Public_Transit_Green_Fuel_Usage', 'Annual_E_Waste_kg']]
    append_to_csv(fact_socio, 'Fact_SocioEconomic.csv')
    
    fact_health = df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Health_Percent', 'Chronic_Disease', 
                            'Hospital_Service_Needed', 'Requires_Hospital_Bed', 'BMI', 'Weekly_Active_Minutes', 
                            'Telemedicine_Consults_Per_Year', 'Health_Insurance_Type']]
    append_to_csv(fact_health, 'Fact_Health_Medical.csv')
    
    fact_edu = df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Edu_Grade', 'Edu_Type', 'School_Type', 
                         'University_Major', 'University_Type']]
    append_to_csv(fact_edu, 'Fact_Education.csv')
    
    family_sizes = df_facts.groupby('Real_Estate_ID').size().reset_index(name='Family_Size')
    fact_env = df_facts[['Real_Estate_ID', 'Report_Year', 'Has_Solar_System', 'Has_AC', 'Uses_Greywater_Recycling', 
                         'Smart_Grid_Tariff_Opt_In', 'Segregates_Waste_Recycling', 'Avg_Monthly_Electricity_KW', 
                         'Avg_Monthly_Gas_m3', 'Avg_Monthly_Water_m3', 'Avg_Monthly_Food_Waste_kg', 'Carbon_Footprint_Index']].drop_duplicates(subset=['Real_Estate_ID'])
    fact_env = pd.merge(fact_env, family_sizes, on='Real_Estate_ID')
    fact_env['Record_ID'] = [str(uuid.uuid4()) for _ in range(len(fact_env))] 
    
    cols = ['Record_ID', 'Report_Year', 'Real_Estate_ID', 'Family_Size', 'Has_Solar_System', 'Has_AC', 'Uses_Greywater_Recycling', 
            'Smart_Grid_Tariff_Opt_In', 'Segregates_Waste_Recycling', 'Avg_Monthly_Electricity_KW', 'Avg_Monthly_Gas_m3', 
            'Avg_Monthly_Water_m3', 'Avg_Monthly_Food_Waste_kg', 'Carbon_Footprint_Index']
    append_to_csv(fact_env[cols], 'Fact_Household_Environment.csv')

# ==========================================
# MAIN SIMULATION ENGINE
# ==========================================
def run_simulation():
    print("Starting Fully Patched & Collision-Free NAC Simulation (2025-2050)...")
    living_population_df = pd.DataFrame()
    
    for year, target_pop in POPULATION_TARGETS.items():
        print(f"\n--- Processing Year {year} ---")
        
        if not living_population_df.empty:
            living_population_df['Current_Age'] += 5
            
            initial_count = len(living_population_df)
            living_population_df = living_population_df[living_population_df['Year_of_Death'] > year]
            print(f"Recorded {initial_count - len(living_population_df):,} deaths since last period.")
            
            # FIXED: Added .any() checks before attempting assignment!
            mask_new_workers = (living_population_df['Current_Age'] >= 22) & (living_population_df['Career'] == 'Unemployed/Student')
            if mask_new_workers.any():
                new_worker_careers = np.random.choice(MAJORS, size=mask_new_workers.sum(), p=MAJOR_PROBS)
                living_population_df.loc[mask_new_workers, 'Career'] = new_worker_careers
                living_population_df.loc[mask_new_workers, 'Income_per_Year_EGP'] = [calculate_skewed_income(c) for c in new_worker_careers]
            
            # FIXED: Added .any() check and prevented double-retiring
            mask_retired = (living_population_df['Current_Age'] >= 65) & (living_population_df['Career'] != 'Retired')
            if mask_retired.any():
                living_population_df.loc[mask_retired, 'Career'] = 'Retired'
                living_population_df.loc[mask_retired, 'Income_per_Year_EGP'] = [calculate_skewed_income('Retired') for _ in range(mask_retired.sum())]
            
            print("Exporting Fact snapshots for surviving population...")
            extract_and_save_relational_data(living_population_df, is_new_batch=False, year=year)
            
        current_pop = len(living_population_df)
        shortfall = target_pop - current_pop
        
        if shortfall > 0:
            print(f"Generating {shortfall:,} new residents to reach {target_pop:,} target...")
            new_batch_df = generate_population_batch(year, shortfall)
            
            print("Exporting Dimensions and Fact snapshots for new arrivals...")
            extract_and_save_relational_data(new_batch_df, is_new_batch=True, year=year)
            
            living_population_df = pd.concat([living_population_df, new_batch_df], ignore_index=True)
                
    print("\nSimulation complete. Database schema created in /nac_relational_dataset folder.")

if __name__ == "__main__":
    run_simulation()

Starting Fully Patched & Collision-Free NAC Simulation (2025-2050)...

--- Processing Year 2025 ---
Generating 350,000 new residents to reach 350,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2030 ---
Recorded 127 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 743,710 new residents to reach 1,050,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2035 ---
Recorded 584 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 994,517 new residents to reach 1,950,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2040 ---
Recorded 2,032 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 1,277,142 new residents to reach 3,100,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2045 ---
Recorded 5,732